In [ ]:
"""
==============================================================================
  MAIN PIPELINE: Graph Feature Effectiveness in Fraud Detection
  Dataset: Sparkov Credit Card Transactions
==============================================================================

This script compares ML model performance across 3 scenarios:
  1. Tabular features only
  2. Graph features only
  3. Tabular + Graph features (combined)

Models: Logistic Regression, Random Forest, XGBoost, LightGBM
Metrics: Precision, Recall, F1-Score

Usage:
  python notebooks/main_pipeline.py
"""

In [2]:
import sys
import os
import time
import warnings

# Use non-interactive backend for matplotlib (must be before any pyplot import)
import matplotlib
matplotlib.use('Agg')

# Add project root to path
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)
os.chdir(project_root)

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from src.data_loader import (
    load_data, downsample, engineer_tabular_features,
    prepare_features, get_raw_columns_for_graph
)
from src.graph_builder import build_heterogeneous_graph, get_graph_stats
from src.graph_features import (
    extract_graph_features, combine_features,
    analyze_graph_features_by_class,
    compute_customer_features, compute_merchant_features
)
from src.model_trainer import (
    train_and_evaluate, create_comparison_table, get_feature_importance
)
from src.visualization import (
    plot_comparison_bars, plot_improvement_bars,
    plot_feature_importance, plot_confusion_matrices,
    plot_class_distribution, plot_graph_feature_analysis,
    print_results_table
)

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================
TRAIN_PATH = "data/SparkovTrain.csv"
TEST_PATH = "data/SparkovTest.csv"
TRAIN_DOWNSAMPLE_SIZE = 50000
TEST_DOWNSAMPLE_SIZE = 20000
RANDOM_STATE = 42
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
# =============================================================================
# SECTION 1: DATA LOADING & EDA
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 1: DATA LOADING & EXPLORATORY DATA ANALYSIS")
print("#" * 70)

start_time = time.time()

df_train_full, df_test_full = load_data(TRAIN_PATH, TEST_PATH)

print(f"\n[INFO] Train columns: {list(df_train_full.columns)}")
print(f"[INFO] Sample data:")
print(df_train_full.head(3).to_string())


######################################################################
#  SECTION 1: DATA LOADING & EXPLORATORY DATA ANALYSIS
######################################################################
[STEP] Loading Sparkov Dataset
[INFO] Train shape: (1296675, 23)
[INFO] Test shape:  (555719, 23)
[INFO] Train fraud: 7506 / 1296675 (0.579%)
[INFO] Test fraud:  2145 / 555719 (0.386%)

[INFO] Train columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']
[INFO] Sample data:
   Unnamed: 0 trans_date_trans_time            cc_num                         merchant       category     amt      first     last gender                        street            city state    zip      lat      long  city_pop                                job         dob                         trans_num   unix_time  merch_l

In [5]:
# =============================================================================
# SECTION 2: DOWNSAMPLING
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 2: DOWNSAMPLING")
print("#" * 70)

y_train_original = df_train_full['is_fraud']

df_train = downsample(df_train_full, target_total=TRAIN_DOWNSAMPLE_SIZE,
                      random_state=RANDOM_STATE)
df_test = downsample(df_test_full, target_total=TEST_DOWNSAMPLE_SIZE,
                     random_state=RANDOM_STATE)

print(f"\n[INFO] After downsampling:")
print(f"  Train: {len(df_train)} ({df_train['is_fraud'].sum()} fraud)")
print(f"  Test:  {len(df_test)} ({df_test['is_fraud'].sum()} fraud)")


######################################################################
#  SECTION 2: DOWNSAMPLING
######################################################################
[INFO] Downsampled: 7506 fraud + 42494 legit = 50000 total
[INFO] New fraud rate: 15.01%
[INFO] Downsampled: 2145 fraud + 17855 legit = 20000 total
[INFO] New fraud rate: 10.72%

[INFO] After downsampling:
  Train: 50000 (7506 fraud)
  Test:  20000 (2145 fraud)


In [6]:
# =============================================================================
# SECTION 3: FEATURE ENGINEERING (TABULAR)
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 3: TABULAR FEATURE ENGINEERING")
print("#" * 70)

# Save raw data for graph construction BEFORE engineering
df_train_raw = df_train.copy()
df_test_raw = df_test.copy()

# Engineer tabular features
df_train_eng = engineer_tabular_features(df_train)
df_test_eng = engineer_tabular_features(df_test)

# Prepare final tabular feature matrices
X_train_tab, X_test_tab, y_train, y_test, tabular_feature_names, scaler = \
    prepare_features(df_train_eng, df_test_eng, scale=True)

print(f"\n[INFO] Tabular features ({len(tabular_feature_names)}):")
print(f"  {tabular_feature_names}")


######################################################################
#  SECTION 3: TABULAR FEATURE ENGINEERING
######################################################################
[INFO] Tabular features: 27 columns
[INFO] X_train shape: (50000, 27), X_test shape: (20000, 27)
[INFO] y_train fraud: 7506, y_test fraud: 2145

[INFO] Tabular features (27):
  ['amt', 'amt_log', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'hour', 'day_of_week', 'month', 'age', 'distance', 'gender_encoded', 'cat_entertainment', 'cat_food_dining', 'cat_gas_transport', 'cat_grocery_net', 'cat_grocery_pos', 'cat_health_fitness', 'cat_home', 'cat_kids_pets', 'cat_misc_net', 'cat_misc_pos', 'cat_personal_care', 'cat_shopping_net', 'cat_shopping_pos', 'cat_travel']


In [7]:
# =============================================================================
# SECTION 4: PHASE 1 - TABULAR ONLY TRAINING
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 4: PHASE 1 - TRAINING WITH TABULAR FEATURES ONLY")
print("#" * 70)

results_tabular, models_tabular = train_and_evaluate(
    X_train_tab, y_train, X_test_tab, y_test,
    scenario_name="Tabular Only"
)


######################################################################
#  SECTION 4: PHASE 1 - TRAINING WITH TABULAR FEATURES ONLY
######################################################################

  Scenario: Tabular Only
  Features: 27 | Train: 50000 | Test: 20000
  Train fraud: 7506 (15.01%)
  Scale pos weight: 5.66

  [Training] Logistic Regression...
    Precision: 0.6554
    Recall:    0.7688
    F1-Score:  0.7076
    Confusion Matrix:
[[16988   867]
 [  496  1649]]

  [Training] Random Forest...
    Precision: 0.9057
    Recall:    0.9497
    F1-Score:  0.9272
    Confusion Matrix:
[[17643   212]
 [  108  2037]]

  [Training] XGBoost...
    Precision: 0.9112
    Recall:    0.9469
    F1-Score:  0.9287
    Confusion Matrix:
[[17657   198]
 [  114  2031]]

  [Training] LightGBM...
    Precision: 0.9208
    Recall:    0.9431
    F1-Score:  0.9318
    Confusion Matrix:
[[17681   174]
 [  122  2023]]


In [8]:
# =============================================================================
# SECTION 5: BUILD HETEROGENEOUS GRAPH
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 5: BUILD HETEROGENEOUS GRAPH FROM TRAINING DATA")
print("#" * 70)

# Prepare raw data for graph construction
df_train_graph = get_raw_columns_for_graph(df_train_raw)
df_test_graph = get_raw_columns_for_graph(df_test_raw)

# Ensure datetime column
df_train_graph['trans_date_trans_time'] = pd.to_datetime(
    df_train_raw['trans_date_trans_time']
)
df_test_graph['trans_date_trans_time'] = pd.to_datetime(
    df_test_raw['trans_date_trans_time']
)

# Build heterogeneous graph from TRAINING DATA ONLY
G = build_heterogeneous_graph(df_train_graph)

# Print detailed graph statistics
stats = get_graph_stats(G)
print(f"\n[INFO] Detailed graph statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")


######################################################################
#  SECTION 5: BUILD HETEROGENEOUS GRAPH FROM TRAINING DATA
######################################################################
[STEP] Building Heterogeneous Graph from Training Data
[INFO] Added 983 Customer nodes
[INFO] Added 693 Merchant nodes
[INFO] Added 14 Category nodes
[INFO] Aggregating Customer-Merchant transaction statistics...


Adding C-M edges: 100%|██████████| 47399/47399 [00:02<00:00, 22950.86it/s]

[INFO] Added 47399 Customer-Merchant edges
[INFO] Added 700 Merchant-Category edges

[INFO] Graph summary:
  Total nodes: 1690
  Total edges: 48099
  Customer nodes: 983
  Merchant nodes: 693
  Category nodes: 14
  Density: 0.033702

[INFO] Detailed graph statistics:
  num_nodes: 1690
  num_edges: 48099
  node_types: {'customer': 983, 'merchant': 693, 'category': 14}
  edge_types: {'transacts_at': 47399, 'belongs_to': 700}
  avg_degree: 56.92189349112426
  max_degree: 168
  min_degree: 6
  density: 0.03370153551872366
  num_connected_components: 1


In [9]:
# =============================================================================
# SECTION 6: EXTRACT GRAPH FEATURES
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 6: EXTRACT GRAPH FEATURES")
print("#" * 70)

# Compute node-level features from training graph (reusable for train & test)
print("\n--- Computing customer features from training graph ---")
customer_features = compute_customer_features(G, df_train_graph)

print("\n--- Computing merchant features from training graph ---")
merchant_features = compute_merchant_features(G, df_train_graph)

# Extract graph features for TRAINING data (fit scaler)
print("\n--- Extracting graph features for TRAINING data ---")
graph_feats_train, graph_scaler = extract_graph_features(
    G, df_train_graph, df_train_graph,
    customer_features=customer_features,
    merchant_features=merchant_features,
    fit_scaler=True
)

# Also extract UNSCALED features for analysis/interpretability
print("\n--- Extracting UNSCALED graph features for analysis ---")
graph_feats_train_unscaled, _ = extract_graph_features(
    G, df_train_graph, df_train_graph,
    customer_features=customer_features,
    merchant_features=merchant_features,
    fit_scaler=False
)

# Extract graph features for TEST data (use training-derived scaler)
print("\n--- Extracting graph features for TEST data ---")
graph_feats_test, _ = extract_graph_features(
    G, df_train_graph, df_test_graph,
    customer_features=customer_features,
    merchant_features=merchant_features,
    scaler=graph_scaler,
    fit_scaler=False
)

# Analyze graph feature discriminative power (on UNSCALED data for interpretability)
print("\n--- Graph Feature Analysis (Training Data - Unscaled) ---")
analysis_df = analyze_graph_features_by_class(graph_feats_train_unscaled, y_train)


######################################################################
#  SECTION 6: EXTRACT GRAPH FEATURES
######################################################################

--- Computing customer features from training graph ---
[INFO] Computing customer-level features...
  [6/12] Computing degree centrality...
  [7/12] Computing PageRank...
  [8/12] Computing community detection (Louvain)...
  [9/12] Computing neighbor fraud ratio...
[INFO] Customer features shape: (983, 8)

--- Computing merchant features from training graph ---
[INFO] Computing merchant-level features...
  [4/12] Computing merchant fraud rate (Bayesian smoothed)...
  [5/12] Computing merchant degree...
  [10/12] Computing shared fraud entity count...
[INFO] Merchant features shape: (693, 8)

--- Extracting graph features for TRAINING data ---
[STEP] Extracting Graph Features
[INFO] Mapping customer features to transactions...
[INFO] Mapping merchant features to transactions...
  [11-12/12] Computing temporal

Temporal features: 100%|██████████| 983/983 [00:00<00:00, 2570.72it/s]


[INFO] Fitted and applied StandardScaler to graph features

[INFO] Total graph features: 18
[INFO] Feature names: ['user_unique_merchants', 'user_txn_frequency', 'user_avg_amt', 'user_std_amt', 'customer_degree_centrality', 'customer_pagerank', 'customer_community_size', 'customer_neighbor_fraud_ratio', 'merchant_fraud_rate', 'merchant_txn_count', 'merchant_total_amt', 'merchant_degree', 'merchant_degree_centrality', 'merchant_pagerank', 'merchant_community_size', 'merchant_shared_fraud_users', 'user_txn_burst', 'user_merchant_sequence_len']
[INFO] Shape: (50000, 18)

--- Extracting UNSCALED graph features for analysis ---
[STEP] Extracting Graph Features
[INFO] Mapping customer features to transactions...
[INFO] Mapping merchant features to transactions...
  [11-12/12] Computing temporal features (window=1h)...


Temporal features: 100%|██████████| 983/983 [00:00<00:00, 2261.87it/s]


[INFO] No scaling applied to graph features

[INFO] Total graph features: 18
[INFO] Feature names: ['user_unique_merchants', 'user_txn_frequency', 'user_avg_amt', 'user_std_amt', 'customer_degree_centrality', 'customer_pagerank', 'customer_community_size', 'customer_neighbor_fraud_ratio', 'merchant_fraud_rate', 'merchant_txn_count', 'merchant_total_amt', 'merchant_degree', 'merchant_degree_centrality', 'merchant_pagerank', 'merchant_community_size', 'merchant_shared_fraud_users', 'user_txn_burst', 'user_merchant_sequence_len']
[INFO] Shape: (50000, 18)

--- Extracting graph features for TEST data ---
[STEP] Extracting Graph Features
[INFO] Mapping customer features to transactions...
[INFO] Mapping merchant features to transactions...
  [11-12/12] Computing temporal features (window=1h)...


Temporal features: 100%|██████████| 923/923 [00:00<00:00, 5116.76it/s]


[INFO] Applied pre-fitted StandardScaler to graph features

[INFO] Total graph features: 18
[INFO] Feature names: ['user_unique_merchants', 'user_txn_frequency', 'user_avg_amt', 'user_std_amt', 'customer_degree_centrality', 'customer_pagerank', 'customer_community_size', 'customer_neighbor_fraud_ratio', 'merchant_fraud_rate', 'merchant_txn_count', 'merchant_total_amt', 'merchant_degree', 'merchant_degree_centrality', 'merchant_pagerank', 'merchant_community_size', 'merchant_shared_fraud_users', 'user_txn_burst', 'user_merchant_sequence_len']
[INFO] Shape: (20000, 18)

--- Graph Feature Analysis (Training Data - Unscaled) ---

  Graph Feature Analysis: Fraud vs Legit
  Feature                             |   Legit Mean |   Fraud Mean |    Ratio
------------------------------------------------------------------------------------------
  user_unique_merchants               |    64.158140 |    50.012523 |    0.780
  user_txn_frequency                  |     1.194716 |     1.693424 |    1.4

In [10]:
# =============================================================================
# SECTION 7: PHASE 2 - GRAPH FEATURES ONLY
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 7: PHASE 2 - TRAINING WITH GRAPH FEATURES ONLY")
print("#" * 70)

graph_feature_names = list(graph_feats_train.columns)
print(f"[INFO] Graph features ({len(graph_feature_names)}): {graph_feature_names}")

results_graph, models_graph = train_and_evaluate(
    graph_feats_train, y_train,
    graph_feats_test, y_test,
    scenario_name="Graph Only"
)


######################################################################
#  SECTION 7: PHASE 2 - TRAINING WITH GRAPH FEATURES ONLY
######################################################################
[INFO] Graph features (18): ['user_unique_merchants', 'user_txn_frequency', 'user_avg_amt', 'user_std_amt', 'customer_degree_centrality', 'customer_pagerank', 'customer_community_size', 'customer_neighbor_fraud_ratio', 'merchant_fraud_rate', 'merchant_txn_count', 'merchant_total_amt', 'merchant_degree', 'merchant_degree_centrality', 'merchant_pagerank', 'merchant_community_size', 'merchant_shared_fraud_users', 'user_txn_burst', 'user_merchant_sequence_len']

  Scenario: Graph Only
  Features: 18 | Train: 50000 | Test: 20000
  Train fraud: 7506 (15.01%)
  Scale pos weight: 5.66

  [Training] Logistic Regression...
    Precision: 0.2907
    Recall:    0.4741
    F1-Score:  0.3604
    Confusion Matrix:
[[15374  2481]
 [ 1128  1017]]

  [Training] Random Forest...
    Precision: 0.1678
    Re

In [11]:
# =============================================================================
# SECTION 8: PHASE 3 - TABULAR + GRAPH (COMBINED)
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 8: PHASE 3 - TRAINING WITH TABULAR + GRAPH FEATURES")
print("#" * 70)

# For the combined scenario, we use ALL graph features.
# The graph features are already Bayesian-smoothed and scaled.
X_train_combined = combine_features(X_train_tab, graph_feats_train)
X_test_combined = combine_features(X_test_tab, graph_feats_test)

combined_feature_names = list(X_train_combined.columns)
print(f"[INFO] Combined features ({len(combined_feature_names)})")

results_combined, models_combined = train_and_evaluate(
    X_train_combined, y_train,
    X_test_combined, y_test,
    scenario_name="Tabular + Graph"
)


######################################################################
#  SECTION 8: PHASE 3 - TRAINING WITH TABULAR + GRAPH FEATURES
######################################################################
[INFO] Combined features: 45 (tabular: 27, graph: 18)
[INFO] Combined features: 45 (tabular: 27, graph: 18)
[INFO] Combined features (45)

  Scenario: Tabular + Graph
  Features: 45 | Train: 50000 | Test: 20000
  Train fraud: 7506 (15.01%)
  Scale pos weight: 5.66

  [Training] Logistic Regression...
    Precision: 0.7235
    Recall:    0.7795
    F1-Score:  0.7504
    Confusion Matrix:
[[17216   639]
 [  473  1672]]

  [Training] Random Forest...
    Precision: 0.8297
    Recall:    0.2862
    F1-Score:  0.4256
    Confusion Matrix:
[[17729   126]
 [ 1531   614]]

  [Training] XGBoost...
    Precision: 0.9234
    Recall:    0.5566
    F1-Score:  0.6946
    Confusion Matrix:
[[17756    99]
 [  951  1194]]

  [Training] LightGBM...
    Precision: 0.9179
    Recall:    0.5944
    F1-Sc

In [12]:
# =============================================================================
# SECTION 9: COMPARISON TABLE
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 9: COMPARISON RESULTS")
print("#" * 70)

all_results = {
    'Tabular Only': results_tabular,
    'Graph Only': results_graph,
    'Tabular + Graph': results_combined
}

comparison_df = create_comparison_table(all_results)

# Print formatted results
print_results_table(comparison_df)

# Save to CSV
comparison_df.to_csv(os.path.join(RESULTS_DIR, "comparison_results.csv"), index=False)
print(f"\n[INFO] Saved results to {RESULTS_DIR}/comparison_results.csv")


######################################################################
#  SECTION 9: COMPARISON RESULTS
######################################################################

  FINAL COMPARISON RESULTS
  Model                  | Scenario                               |    Prec |  Recall |      F1
-------------------------------------------------------------------------------------
  Logistic Regression    | Tabular Only                           |  0.6554 |  0.7688 |  0.7076
  Logistic Regression    | Graph Only                             |  0.2907 |  0.4741 |  0.3604
  Logistic Regression    | Tabular + Graph                        |  0.7235 |  0.7795 |  0.7504
  Logistic Regression    | [Delta] Improvement (Combined vs Tabular) | +0.0681 | +0.0107 | +0.0429
-------------------------------------------------------------------------------------
  Random Forest          | Tabular Only                           |  0.9057 |  0.9497 |  0.9272
  Random Forest          | Graph Only        

In [13]:
# =============================================================================
# SECTION 10: VISUALIZATION
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 10: VISUALIZATION")
print("#" * 70)

# 10.1 Class distribution
plot_class_distribution(
    y_train_original, y_train, y_test,
    save_path=os.path.join(RESULTS_DIR, "class_distribution.png")
)

# 10.2 Comparison bar chart
plot_comparison_bars(
    comparison_df,
    save_path=os.path.join(RESULTS_DIR, "comparison_bars.png")
)

# 10.3 Improvement chart
plot_improvement_bars(
    comparison_df,
    save_path=os.path.join(RESULTS_DIR, "improvement_bars.png")
)

# 10.4 Feature importance for Combined scenario
importance_combined = get_feature_importance(
    models_combined, combined_feature_names, top_n=20
)
plot_feature_importance(
    importance_combined,
    scenario_name="Tabular + Graph Combined",
    save_path=os.path.join(RESULTS_DIR, "feature_importance_combined.png")
)

# 10.5 Feature importance for Graph Only scenario
importance_graph = get_feature_importance(
    models_graph, graph_feature_names, top_n=20
)
plot_feature_importance(
    importance_graph,
    scenario_name="Graph Only",
    save_path=os.path.join(RESULTS_DIR, "feature_importance_graph.png")
)

# 10.6 Confusion matrices
plot_confusion_matrices(
    all_results,
    save_path=os.path.join(RESULTS_DIR, "confusion_matrices.png")
)

# 10.7 Graph feature analysis
plot_graph_feature_analysis(
    analysis_df,
    save_path=os.path.join(RESULTS_DIR, "graph_feature_analysis.png")
)


######################################################################
#  SECTION 10: VISUALIZATION
######################################################################
[INFO] Saved class distribution to results\class_distribution.png
[INFO] Saved comparison chart to results\comparison_bars.png
[INFO] Saved improvement chart to results\improvement_bars.png
[INFO] Saved feature importance to results\feature_importance_combined.png
[INFO] Saved feature importance to results\feature_importance_graph.png
[INFO] Saved confusion matrices to results\confusion_matrices.png
[INFO] Saved graph feature analysis to results\graph_feature_analysis.png


In [14]:
# =============================================================================
# SECTION 11: SUMMARY
# =============================================================================
print("\n" + "#" * 70)
print("#  SECTION 11: FINAL SUMMARY")
print("#" * 70)

elapsed = time.time() - start_time
print(f"\n[INFO] Total execution time: {elapsed:.1f}s ({elapsed/60:.1f} minutes)")
print(f"\n[INFO] Results saved to {RESULTS_DIR}/")
print(f"  - comparison_results.csv")
print(f"  - comparison_bars.png")
print(f"  - improvement_bars.png")
print(f"  - feature_importance_combined.png")
print(f"  - feature_importance_graph.png")
print(f"  - confusion_matrices.png")
print(f"  - class_distribution.png")
print(f"  - graph_feature_analysis.png")

# Print key findings
print("\n" + "=" * 70)
print("  KEY FINDINGS")
print("=" * 70)

for model in ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']:
    f1_tab = results_tabular[model]['f1']
    f1_graph = results_graph[model]['f1']
    f1_comb = results_combined[model]['f1']
    delta = f1_comb - f1_tab

    arrow = "[UP]" if delta > 0 else ("[DOWN]" if delta < 0 else "[=]")
    print(f"  {model:<22}: F1 Tabular={f1_tab:.4f} | Graph={f1_graph:.4f} | "
          f"Combined={f1_comb:.4f} | {arrow} {delta:+.4f}")

print("\n" + "=" * 70)
print("  CONCLUSION")
print("=" * 70)

# Auto-generate conclusion
avg_improvement = np.mean([
    results_combined[m]['f1'] - results_tabular[m]['f1']
    for m in ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM']
])

if avg_improvement > 0.01:
    print(f"  [OK] Graph features IMPROVED fraud detection performance!")
    print(f"     Average F1-Score improvement: {avg_improvement:+.4f}")
elif avg_improvement > -0.01:
    print(f"  [~] Graph features had marginal effect on performance.")
    print(f"     Average F1-Score change: {avg_improvement:+.4f}")
else:
    print(f"  [!] Graph features did not improve performance in this configuration.")
    print(f"     Average F1-Score change: {avg_improvement:+.4f}")

print(f"\n  Best graph-only features (most discriminative):")
top_feats = analysis_df.sort_values('ratio', ascending=False).head(5)
for _, row in top_feats.iterrows():
    print(f"    - {row['feature']}: fraud/legit ratio = {row['ratio']:.3f}")

print("\n" + "#" * 70)
print("#  DONE!")
print("#" * 70)


######################################################################
#  SECTION 11: FINAL SUMMARY
######################################################################

[INFO] Total execution time: 236.0s (3.9 minutes)

[INFO] Results saved to results/
  - comparison_results.csv
  - comparison_bars.png
  - improvement_bars.png
  - feature_importance_combined.png
  - feature_importance_graph.png
  - confusion_matrices.png
  - class_distribution.png
  - graph_feature_analysis.png

  KEY FINDINGS
  Logistic Regression   : F1 Tabular=0.7076 | Graph=0.3604 | Combined=0.7504 | [UP] +0.0429
  Random Forest         : F1 Tabular=0.9272 | Graph=0.1572 | Combined=0.4256 | [DOWN] -0.5015
  XGBoost               : F1 Tabular=0.9287 | Graph=0.1216 | Combined=0.6946 | [DOWN] -0.2341
  LightGBM              : F1 Tabular=0.9318 | Graph=0.2448 | Combined=0.7216 | [DOWN] -0.2103

  CONCLUSION
  [!] Graph features did not improve performance in this configuration.
     Average F1-Score change: -0.2257